# LLM Evaluation

Build a small DGIdb/PubMed binary drug-gene pair answer set, optionally run Wags-LLM extraction, and score the predictions.

Prior to running the model, run:

`aws sso login --profile dev-account`

In [9]:
import json
from pathlib import Path

import pandas as pd
from answer_sheet import task_payloads_from_pmids, task_pmids_from_abstracts
from curation import build_curated_answer_sheet
from dgidb import candidate_pair_frame, load_cached_dgidb_source
from pubmed import fetch_pubmed_abstracts, filter_interactions_with_abstracts
from scoring import score_predictions
from utils import out_path
from wags import prediction_status_frame, run_wags_predictions, save_predictions


In [2]:
ENABLE_LLM_CALLS = True
N_PMIDS = 100
STRICT_DRUG_TOKEN_FILTER = True

OUT_DIR = Path("outputs")
CACHE_DIR = OUT_DIR / "dgidb_cache"
OUT_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

DGIDB_GRAPHQL_URL = "https://dgidb.org/api/graphql"
PUBMED_EFETCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

MODEL_KEY = "claude-sonnet-4-6"
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
AWS_PROFILE_NAME = "dev-account"
BEDROCK_REGION = "us-east-2"
PROMPT_NAME = "dgidb_pubmed_drug_gene_pair_extraction"
PROMPT_VERSION = "v2026-06-22"
PROMPT_VERSION_SLUG = PROMPT_VERSION.replace("/", "_").replace(":", "_")

PATHS = {
    "dgidb_source_cache": CACHE_DIR / "publication_interactions.csv",
    "pubmed_abstract_cache": OUT_DIR / "pubmed_abstracts.csv",
    "answer_sheet": out_path(OUT_DIR, "answer_sheet", n_pmids=N_PMIDS),
    "curated_literature_set": out_path(
        OUT_DIR, "curated_literature_set", n_pmids=N_PMIDS
    ),
    "curation_pmid_audit": out_path(OUT_DIR, "curation_pmid_audit", n_pmids=N_PMIDS),
    "curation_summary": OUT_DIR / "curation_summary.csv",
    "pmid_tasks": out_path(OUT_DIR, "pmid_tasks_with_abstracts", n_pmids=N_PMIDS),
    "prompt_payloads": out_path(
        OUT_DIR, "wags_prompt_payloads", suffix="json", n_pmids=N_PMIDS
    ),
    "predictions": OUT_DIR
    / f"predictions_{MODEL_KEY}_{PROMPT_VERSION_SLUG}_{N_PMIDS}.json",
    "metrics": OUT_DIR / f"metrics_{MODEL_KEY}_{PROMPT_VERSION_SLUG}_{N_PMIDS}.csv",
}
PATHS


{'dgidb_source_cache': PosixPath('outputs/dgidb_cache/publication_interactions.csv'),
 'pubmed_abstract_cache': PosixPath('outputs/pubmed_abstracts.csv'),
 'answer_sheet': PosixPath('outputs/answer_sheet_100.csv'),
 'curated_literature_set': PosixPath('outputs/curated_literature_set_100.csv'),
 'curation_pmid_audit': PosixPath('outputs/curation_pmid_audit_100.csv'),
 'curation_summary': PosixPath('outputs/curation_summary.csv'),
 'pmid_tasks': PosixPath('outputs/pmid_tasks_with_abstracts_100.csv'),
 'prompt_payloads': PosixPath('outputs/wags_prompt_payloads_100.json'),
 'predictions': PosixPath('outputs/predictions_claude-sonnet-4-6_v2026-06-22_100.json'),
 'metrics': PosixPath('outputs/metrics_claude-sonnet-4-6_v2026-06-22_100.csv')}

## DGIdb candidate pairs

In [3]:
source_df = load_cached_dgidb_source(
    source_cache_path=PATHS["dgidb_source_cache"],
    graphql_url=DGIDB_GRAPHQL_URL,
    return_source=True,
)
eligible_df = candidate_pair_frame(source_df)
eligible_df.head()


,pmid,drug_name,gene_name
0,10022508,DIHYDROSPHINGOSINE,IL6
1,10022737,BACILLUS CALMETTE-GUERIN ANTIGEN,TNF
2,10022737,INTERFERON ALFA-2B,IL6
3,10022751,TOLBUTAMIDE,CYP2C19
4,10023678,IL-6,CDK2


## PubMed abstracts

In [4]:
eligible_abstracts_df = fetch_pubmed_abstracts(
    sorted(eligible_df["pmid"].astype(str).unique()),
    cache_path=PATHS["pubmed_abstract_cache"],
    efetch_url=PUBMED_EFETCH_URL,
)

eligible_df, eligible_abstracts_df, abstract_filter_stats = (
    filter_interactions_with_abstracts(
        eligible_df,
        eligible_abstracts_df,
        min_pmids=N_PMIDS,
        strict_drug_token_filter=STRICT_DRUG_TOKEN_FILTER,
    )
)
pd.DataFrame([abstract_filter_stats])


,pmids_before_abstract_filter,pmids_removed_without_abstract,pmids_after_abstract_filter,pmids_removed_without_recoverable_pair,pmids_after_pair_filter,eligible_pair_rows_after_filter
0,12285,267,12018,7539,4479,6190


## Curated answer sheet and task payloads


In [5]:
answer_sheet, curated_literature_set, curation_pmid_audit, curation_summary = (
    build_curated_answer_sheet(
        source_df,
        eligible_abstracts_df,
        n_pmids=N_PMIDS,
        strict_drug_token_filter=STRICT_DRUG_TOKEN_FILTER,
    )
)
selected_pmids = curation_pmid_audit["pmid"].astype(str).tolist()

task_pmids = task_pmids_from_abstracts(selected_pmids, eligible_abstracts_df)
task_payloads = task_payloads_from_pmids(task_pmids)

answer_sheet.to_csv(PATHS["answer_sheet"], index=False)
curated_literature_set.to_csv(PATHS["curated_literature_set"], index=False)
curation_pmid_audit.to_csv(PATHS["curation_pmid_audit"], index=False)
pd.DataFrame([{**abstract_filter_stats, **curation_summary}]).to_csv(
    PATHS["curation_summary"], index=False
)
task_pmids.to_csv(PATHS["pmid_tasks"], index=False)
PATHS["prompt_payloads"].write_text(
    json.dumps(task_payloads, indent=2, ensure_ascii=False) + "\n"
)

answer_sheet.head()


,pmid,drug_name,gene_name
0,22608338,DABRAFENIB,BRAF
1,22735384,DABRAFENIB,BRAF
2,38756640,DABRAFENIB,BRAF
3,24508103,VEMURAFENIB,BRAF
4,27460442,VEMURAFENIB,BRAF


## Wags-LLM predictions

In [10]:
predictions = run_wags_predictions(
    task_payloads,
    predictions_path=PATHS["predictions"],
    enable_llm_calls=ENABLE_LLM_CALLS,
    model_key=MODEL_KEY,
    model_id=MODEL_ID,
    prompt_name=PROMPT_NAME,
    prompt_version=PROMPT_VERSION,
    aws_profile_name=AWS_PROFILE_NAME,
    bedrock_region=BEDROCK_REGION,
    max_tokens=2048,
    temperature=0.0,
    allowed_pmids=task_pmids["pmid"].astype(str),
)
if predictions:
    save_predictions(predictions, PATHS["predictions"])

prediction_status_frame(predictions)

,pmid,successful,parsed_interactions,error
0,11158396,True,2,
1,11217909,True,1,
2,12832683,True,1,
3,15236790,True,1,
4,17827387,True,3,
...,...,...,...,...
95,38598794,True,1,
96,38756640,True,4,
97,39082679,True,2,
98,9551410,True,1,


## Score predictions

In [11]:
metrics_df = score_predictions(
    answer_sheet,
    predictions_path=PATHS["predictions"],
    metrics_path=PATHS["metrics"],
)
metrics_df

,expected,predicted,true_positives,false_positives,false_negatives,precision,recall,f1,scored_pmids,sampled_answer_pmids,prediction_records,successful_prediction_records,error_records
0,107,204,99,105,8,0.485294,0.925234,0.636656,100,100,100,100,0


## Generated outputs

In [13]:
metrics_df.to_csv("test.csv")

In [12]:
pd.DataFrame(
    {"name": name, "path": str(path), "exists": Path(path).exists()}
    for name, path in PATHS.items()
)

,name,path,exists
0,dgidb_source_cache,outputs/dgidb_cache/publication_interactions.csv,True
1,pubmed_abstract_cache,outputs/pubmed_abstracts.csv,True
2,answer_sheet,outputs/answer_sheet_100.csv,True
3,curated_literature_set,outputs/curated_literature_set_100.csv,True
4,curation_pmid_audit,outputs/curation_pmid_audit_100.csv,True
5,curation_summary,outputs/curation_summary.csv,True
6,pmid_tasks,outputs/pmid_tasks_with_abstracts_100.csv,True
7,prompt_payloads,outputs/wags_prompt_payloads_100.json,True
8,predictions,outputs/predictions_claude-sonnet-4-6_v2026-06...,True
9,metrics,outputs/metrics_claude-sonnet-4-6_v2026-06-22_...,True
